# 19. Ragas 정량 평가 — Faithfulness · Relevancy · Context P·R
> Day 4 · 22H · 소요 약 50분

## 학습 목표

- RAG 시스템의 4대 평가 메트릭(Faithfulness, Answer Relevancy, Context Precision, Context Recall)을 설명할 수 있다.
- Ragas로 본인 SQL 에이전트를 정량적으로 평가할 수 있다.
- 낮은 점수의 원인을 진단하고 프롬프트를 개선할 수 있다.
- Before/After 비교 차트로 개선 효과를 시각화해 발표 슬라이드에 활용한다.

> **전제 노트북:** 17번 에이전트를 self-contained 로 다시 이식 후 Ragas 입력 형식(question / answer / contexts / ground_truth) 으로 변환합니다.
> **필요 키:** `OPENAI_API_KEY`, `NEON_DSN`. `LANGSMITH_API_KEY` 는 **선택** (있으면 Ragas 내부 판정 호출도 트레이싱됨).

### API 비용 주의

Ragas 의 각 메트릭은 내부적으로 **LLM 판정자(judge)** 를 여러 번 호출합니다. 4 메트릭 × 10 질문 = 약 40~80회 추가 LLM 호출이 발생하며, 판정 LLM을 따로 지정하지 않으면 비용이 크게 오를 수 있습니다. 이 노트북은 **`gpt-4o-mini` 를 판정 LLM으로 명시 주입** 하여 비용을 약 1/10 수준으로 낮춥니다. 처음 실습할 때는 질문 수를 3~5개로 줄여서 파이프라인부터 확인하세요.

### 버전 핀

Ragas 는 0.1 → 0.2 에서 필드 이름이 바뀌었습니다. 본 노트북은 **0.1 계열** (`contexts`, `ground_truth`, `report.to_pandas()`) 기준입니다:

```
pip install "ragas>=0.1.17,<0.2" "datasets>=2.16,<3"
```

In [ ]:
%pip install -q "ragas>=0.1.17,<0.2" "datasets>=2.16,<3" langgraph langchain langchain-openai langsmith sqlalchemy psycopg2-binary sqlparse pandas matplotlib tabulate

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os


def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")


# OpenAI 키와 Neon DSN 은 필수 — Ragas 가 LLM 판정자를 부르고, 우리 에이전트가 DB 를 조회합니다.
_load_secret("OPENAI_API_KEY", required=True)
_load_secret("NEON_DSN", required=True)
# LangSmith 는 선택 — 키가 있으면 Ragas 의 내부 판정 호출까지 트레이싱되어 디버깅이 한결 편해집니다.
_load_secret("LANGSMITH_API_KEY", required=False)
if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGCHAIN_PROJECT", "sql-agent-day4-ragas")
    os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]

print("Environment ready.")
print(f"  LangSmith tracing = {os.environ.get('LANGCHAIN_TRACING_V2', 'off')}")

## 1. 4대 메트릭 요약

| 메트릭 | 묻는 것 | 낮을 때 원인 | 튜닝 방향 |
|---|---|---|---|
| **Faithfulness** | 답변이 컨텍스트에 근거하는가 (= 지어내지 않았는가) | 할루시네이션, LLM 사전지식 유출 | 프롬프트에 "컨텍스트 밖 정보 금지" 명시, temperature=0 |
| **Answer Relevancy** | 답변이 질문에 직접 답하는가 | 관련 없는 답, 일반론 | "질문에 직접 답하세요" 규칙, 불필요한 배경 생략 |
| **Context Precision** | 검색된 컨텍스트 중 실제 기여한 비율 | 불필요한 문서가 많음 | Re-rank, 메타데이터 필터, TopK↓ |
| **Context Recall** | 정답에 필요한 정보를 컨텍스트가 담는가 | 필요한 문서 누락 | TopK↑, 하이브리드 검색, 스키마 COMMENT 보강 |

### SQL 에이전트에서의 해석 주의

Context Precision/Recall은 **여러 문서를 검색하는 순수 RAG** 를 전제로 설계되었습니다. 우리 에이전트는 `contexts` 에 "SQL + 결과" 한 덩어리만 들어가므로:

- **Context Precision**: 거의 항상 1에 가깝게 나옵니다 (문서가 1개뿐).
- **Context Recall**: SQL 결과가 ground_truth 의 사실들을 포함하는지를 봅니다 → 사실상 **"SQL 생성 품질"의 간접 지표**.

발표에서는 이 한계를 언급하면 "평가의 한계까지 인지" 한 것으로 높이 평가됩니다.

## 2. 17번 에이전트 재이식 (self-contained)

NB18 과 동일하게 17번 에이전트를 이식합니다. 로직 / 상태 / 노드 / 재시도 분기 모두 17번과 1:1 일치.

In [ ]:
import re
from typing import TypedDict, Optional, List, Dict, Any

from sqlalchemy import create_engine, inspect, text
import pandas as pd

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- DB 엔진 (읽기 전용 모드 시도) ---
try:
    engine = create_engine(
        os.environ["NEON_DSN"],
        connect_args={"options": "-c default_transaction_read_only=on"},
        pool_pre_ping=True,
    )
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Engine connected in read-only mode.")
except Exception as e:
    print(f"[WARN] read-only 옵션 실패 → 일반 모드로 재연결: {e}")
    engine = create_engine(os.environ["NEON_DSN"], pool_pre_ping=True)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Engine connected (non-RO).")


In [ ]:
def collect_schema(engine, tables=None) -> str:
    """DB 스키마를 LLM 프롬프트용 DDL 텍스트로 변환 (NB17과 동일 로직)."""
    inspector = inspect(engine)
    if tables is None:
        tables = inspector.get_table_names()

    parts = []
    for table in tables:
        columns = inspector.get_columns(table)
        fks = inspector.get_foreign_keys(table)

        col_lines = []
        for col in columns:
            nullable = "" if col["nullable"] else " NOT NULL"
            col_lines.append(f"    {col['name']} {col['type']}{nullable}")

        fk_lines = []
        for fk in fks:
            fk_lines.append(
                f"    FOREIGN KEY ({', '.join(fk['constrained_columns'])}) "
                f"REFERENCES {fk['referred_table']}({', '.join(fk['referred_columns'])})"
            )

        ddl = f"CREATE TABLE {table} (\n"
        ddl += ",\n".join(col_lines)
        if fk_lines:
            ddl += ",\n" + ",\n".join(fk_lines)
        ddl += "\n);"

        try:
            with engine.connect() as conn:
                comments = conn.execute(
                    text("""
                        SELECT a.attname,
                               col_description(c.oid, a.attnum) AS comment
                        FROM pg_class c
                        JOIN pg_namespace n ON n.oid = c.relnamespace
                        JOIN pg_attribute a ON a.attrelid = c.oid
                        WHERE c.relname = :table
                          AND n.nspname = 'public'
                          AND a.attnum > 0
                          AND NOT a.attisdropped
                        ORDER BY a.attnum
                    """),
                    {"table": table},
                ).fetchall()
            for col_name, comment in comments:
                if comment:
                    ddl += f"\n-- {table}.{col_name}: {comment}"
        except Exception:
            pass

        parts.append(ddl)
    return "\n\n".join(parts)


TABLES = ["patients", "doctors", "visits", "diagnoses", "departments"]
_available = set(inspect(engine).get_table_names())
TABLES = [t for t in TABLES if t in _available]
if not TABLES:
    TABLES = list(_available)[:10]
print(f"Tables to include in schema: {TABLES}")

SCHEMA = collect_schema(engine, TABLES)
print(f"\nSchema text length: {len(SCHEMA)} chars")


In [ ]:
class AgentState(TypedDict, total=False):
    question: str
    sql: str
    result: List[Dict[str, Any]]
    result_md: str
    answer: str
    error: Optional[str]
    retry_count: int


BLOCKED_SQL_PATTERN = re.compile(
    r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE|GRANT|REVOKE)\b",
    re.IGNORECASE,
)


def is_safe_sql(sql: str) -> tuple[bool, str]:
    match = BLOCKED_SQL_PATTERN.search(sql)
    if match:
        return False, f"보안 위반: '{match.group()}' 명령은 허용되지 않습니다."
    return True, ""


def inject_limit(sql: str, cap: int = 1000) -> str:
    stripped = sql.strip().rstrip(";")
    if re.search(r"\bLIMIT\s+\d+\b", stripped, re.IGNORECASE):
        return stripped
    return f"{stripped}\nLIMIT {cap}"


def strip_sql_fences(sql: str) -> str:
    sql = re.sub(r"```sql\s*", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"```\s*", "", sql)
    return sql.strip()


In [ ]:
sql_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
answer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)


SQL_GEN_TEMPLATE = ChatPromptTemplate.from_template(
    """당신은 PostgreSQL 전문가입니다. 아래 스키마를 참고하여 질문에 대한 SQL 하나를 작성하세요.

## 스키마
{schema}

## 규칙
- SELECT 문만 작성. DML/DDL 금지.
- visits.status = 'completed' 만 유효한 진료로 간주.
- 나이 = EXTRACT(YEAR FROM AGE(birth_date))
- 결과가 많을 가능성이 있으면 LIMIT 100 이하를 권장.
- 설명 없이 **SQL 만** 반환.
{error_feedback}

## 질문
{question}

SQL:
"""
)

sql_gen_chain = SQL_GEN_TEMPLATE | sql_llm | StrOutputParser()


def generate_sql(state: AgentState) -> dict:
    """노드 1 — 질문 → SQL. 이전 에러가 있으면 피드백으로 재생성."""
    error_feedback = ""
    if state.get("error"):
        error_feedback = (
            "\n## 직전 시도의 실패\n"
            f"- 실패 SQL:\n{state.get('sql', '')}\n"
            f"- 에러: {state['error']}\n"
            "이 에러를 피해서 SQL 을 다시 작성하세요."
        )

    raw = sql_gen_chain.invoke({
        "schema": SCHEMA,
        "question": state["question"],
        "error_feedback": error_feedback,
    })
    sql = strip_sql_fences(raw)
    return {
        "sql": sql,
        "retry_count": state.get("retry_count", 0) + 1,
    }


def execute_sql(state: AgentState) -> dict:
    """노드 2 — SQL 실행 + 결과 Markdown 화."""
    sql = state.get("sql", "")
    if not sql:
        return {"error": "실행할 SQL 이 없습니다.", "result": [], "result_md": ""}

    ok, reason = is_safe_sql(sql)
    if not ok:
        return {"error": reason, "result": [], "result_md": ""}

    safe_sql = inject_limit(sql, cap=1000)
    try:
        df = pd.read_sql(text(safe_sql), engine)
        if df.empty:
            return {
                "result": [],
                "result_md": "(결과 없음 — 조건을 다시 확인하세요)",
                "error": "",
                "sql": safe_sql,
            }
        result_rows = df.head(50).to_dict(orient="records")
        md_table = df.head(50).to_markdown(index=False)
        if len(df) > 50:
            md_table += f"\n\n... 외 {len(df) - 50}행"
        return {
            "result": result_rows,
            "result_md": md_table,
            "error": "",
            "sql": safe_sql,
        }
    except Exception as e:
        return {
            "error": f"SQL 실행 오류: {type(e).__name__}: {str(e)[:300]}",
            "result": [],
            "result_md": "",
        }


def validate_sql(state: AgentState) -> dict:
    """노드 3 — 단순 검증. 에러가 있으면 그대로 두고 분기 함수가 retry 결정."""
    if state.get("error"):
        return {}
    return {"error": ""}


ANSWER_TEMPLATE = ChatPromptTemplate.from_template(
    """다음 SQL 실행 결과를 바탕으로 질문에 한국어로 답변하세요.

## 질문
{question}

## 실행한 SQL
{sql}

## 결과 (Markdown 표)
{result_md}

## 규칙
- 2-3 문장으로 핵심만 요약.
- 숫자에 천 단위 구분자(쉼표) 사용.
- 결과가 비어 있으면 "해당 조건에 맞는 데이터가 없습니다." 로 시작하는 안내.
- 추측 금지 — 표에 없는 수치는 언급하지 말 것.
"""
)

answer_chain = ANSWER_TEMPLATE | answer_llm | StrOutputParser()


def generate_answer(state: AgentState) -> dict:
    """노드 4 — 결과 → 자연어 답변."""
    if state.get("error") and state.get("retry_count", 0) >= 3:
        return {
            "answer": (
                "죄송합니다. 질문에 답변하지 못했습니다.\n"
                f"- 마지막 에러: {state['error']}\n"
                "- 질문을 더 구체적으로 다시 물어봐 주세요."
            )
        }
    ans = answer_chain.invoke({
        "question": state.get("question", ""),
        "sql": state.get("sql", ""),
        "result_md": (state.get("result_md") or "(결과 없음)")[:1500],
    })
    return {"answer": ans.strip()}


MAX_RETRIES = 3


def should_retry(state: AgentState) -> str:
    if not state.get("error"):
        return "answer"
    if state.get("retry_count", 0) >= MAX_RETRIES:
        return "giveup"
    return "retry"


In [ ]:
graph = StateGraph(AgentState)
graph.add_node("generate_sql", generate_sql)
graph.add_node("execute_sql", execute_sql)
graph.add_node("validate_sql", validate_sql)
graph.add_node("generate_answer", generate_answer)

graph.set_entry_point("generate_sql")
graph.add_edge("generate_sql", "execute_sql")
graph.add_edge("execute_sql", "validate_sql")
graph.add_conditional_edges(
    "validate_sql",
    should_retry,
    {
        "answer": "generate_answer",
        "giveup": "generate_answer",
        "retry":  "generate_sql",
    },
)
graph.add_edge("generate_answer", END)

agent = graph.compile()
print("Agent compiled.")

def _initial_state(question: str) -> dict:
    return {
        "question": question,
        "sql": "",
        "result": [],
        "result_md": "",
        "answer": "",
        "error": "",
        "retry_count": 0,
    }


## 3. 평가 데이터 수집 — 10개 질문 실행

Ragas 는 4필드 (`question`, `answer`, `contexts`, `ground_truth`) 의 DataFrame을 요구합니다. 먼저 에이전트를 10번 돌려 답변을 수집한 뒤, SQL + SQL 결과를 `contexts` 로 엮고 사전에 준비된 ground_truth 사전으로 정답을 붙입니다.

In [ ]:
questions = [
    "전체 환자 수는?",
    "남성 환자 중 40세 이상은 몇 명?",
    "진료과별 의사 수를 보여줘",
    "지난달 완료 진료 건수는?",
    "응급 진료 평균 비용은?",
    "가장 많이 방문한 환자 Top 3는?",
    "중증 진단을 받은 환자 이름은?",
    "2026년 월별 방문 수 추이는?",
    "내과 의사 중 급여 최고는?",
    "혈액형별 환자 분포는?",
]

results = []
for i, q in enumerate(questions):
    state = agent.invoke(_initial_state(q))
    results.append({
        "question":  q,
        "sql":       state.get("sql", ""),
        "result_md": state.get("result_md", ""),
        "answer":    state.get("answer", ""),
        "error":     state.get("error", ""),
        "retries":   state.get("retry_count", 0),
    })
    ok = bool(state.get("answer")) and not state.get("error")
    print(f"[{i+1:2d}/{len(questions)}] {'OK' if ok else 'FAIL'} (retries={state.get('retry_count',0)}) — {q}")

print(f"\nCollected {len(results)} agent runs.")

In [ ]:
# Ragas 가 요구하는 4개 컬럼: question / answer / contexts / ground_truth.
# - question     : 사용자 질문
# - answer       : 우리 에이전트의 답변
# - contexts     : "에이전트가 답변할 때 참조한 자료" — 여기서는 SQL + 결과 표 한 덩어리
# - ground_truth : 사람이 검토해 만든 정답 기준선 (Context Recall 채점에 사용)

# 변수명에 _dict 가 붙은 이유: 18번 노트북의 ground_truths(list) 와 같은 셀에서 공존할 때 충돌 방지.
ground_truths_dict = {
    "전체 환자 수는?":                    "전체 환자 수는 30명입니다.",
    "남성 환자 중 40세 이상은 몇 명?":    "남성 환자 중 40세 이상은 약 7명입니다.",
    "진료과별 의사 수를 보여줘":          "내과 3명, 외과 3명, 소아과 3명, 정형외과 2명, 피부과 2명, 신경과 3명, 산부인과 2명, 안과 2명입니다.",
    "지난달 완료 진료 건수는?":          "지난달 완료된 진료 건수를 보여줍니다.",
    "응급 진료 평균 비용은?":             "응급 진료의 평균 비용은 약 300,000원입니다.",
    "가장 많이 방문한 환자 Top 3는?":    "홍길동, 이준석, 강현우 등이 가장 많이 방문한 환자입니다.",
    "중증 진단을 받은 환자 이름은?":     "급성 충수염, 담낭결석, 뇌진탕 등 중증 진단을 받은 환자 목록입니다.",
    "2026년 월별 방문 수 추이는?":       "2026년 1월부터 4월까지 월별 방문 수 추이를 보여줍니다.",
    "내과 의사 중 급여 최고는?":          "내과 의사 중 김철수가 월 8,500,000원으로 가장 높습니다.",
    "혈액형별 환자 분포는?":              "A형, B형, O형, AB형 각각의 환자 수 분포입니다.",
}


def _build_eval_data(rows: list[dict]) -> dict:
    """에이전트 실행 결과 리스트 → Ragas 0.1 입력 형식 (4개 리스트) 로 변환."""
    # Ragas 는 dict[str, list] 형식의 입력을 받기 때문에 4개 빈 리스트로 시작.
    eval_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    for r in rows:
        q = r["question"]
        eval_data["question"].append(q)
        eval_data["answer"].append(r.get("answer", ""))
        # contexts 는 "리스트의 리스트" 형식을 요구합니다 (한 질문에 여러 문서를 줄 수도 있게).
        # SQL 텍스트와 결과 표를 합쳐 한 덩어리로 넣어 Ragas 가 답변과의 일관성을 채점하게 한다.
        ctx = f"SQL: {r.get('sql', '')}\n결과:\n{r.get('result_md', '')}"
        eval_data["contexts"].append([ctx])  # 한 번 더 [] 로 감싸 list[list[str]] 를 만든다
        # 사전에 등록되지 않은 질문은 빈 문자열을 ground_truth 로 — Recall 이 0 으로 나오므로 주의.
        eval_data["ground_truth"].append(ground_truths_dict.get(q, ""))
    return eval_data


eval_data = _build_eval_data(results)
print(f"Ragas 입력 데이터 준비: {len(eval_data['question'])} 건")
print(f"샘플: {eval_data['question'][0]} -> {eval_data['answer'][0][:60]}...")

## 4. Ragas 평가 실행

판정 LLM을 `gpt-4o-mini` 로 주입해 비용을 낮춥니다. 평가 수행 시간은 질문 수 × 4 메트릭 × 수 초 정도 = **대략 2~5분**. 진행 로그가 멈춘 듯 보여도 기다리세요.

In [ ]:
# Ragas 평가 본 셀 — 4개 메트릭으로 한 번에 채점합니다.
from ragas import evaluate
from ragas.metrics import (
    faithfulness,        # 답변이 컨텍스트에 충실한가 (할루시네이션 억제)
    answer_relevancy,    # 답변이 질문의 핵심에 닿는가
    context_precision,   # 검색된 컨텍스트가 실제로 쓸모 있었는가
    context_recall,      # 정답에 필요한 사실들이 컨텍스트에 다 있는가
)
from ragas.llms import LangchainLLMWrapper           # LangChain LLM 을 Ragas 형식으로 감싸 주는 래퍼
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings
from datasets import Dataset                          # HuggingFace datasets 라이브러리

# dict → Dataset 변환. Ragas 가 내부적으로 이 형식으로 작업합니다.
eval_dataset = Dataset.from_dict(eval_data)

# 판정자(Judge) LLM = Ragas 가 메트릭 채점할 때 사용할 LLM. 큰 모델을 쓰면 평가가 더 엄격해지지만 비용도 비례.
# gpt-4o-mini 로 명시 주입하면 기본값(gpt-4) 대비 비용을 약 1/10 수준으로 절감할 수 있습니다.
judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

print("Ragas 평가 실행 중... (약 2~5분)")
report = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm,           # 판정자 LLM 주입 (생략 시 OpenAI 기본 모델 사용 → 비싸짐)
    embeddings=judge_emb,    # 일부 메트릭에서 임베딩 유사도 비교용
)

# Ragas 결과를 DataFrame 으로 변환 → 메트릭별 평균 점수 계산.
df_report = report.to_pandas()
# 컬럼 이름 호환: 0.1 / 0.2 사이에 일부 변경됐을 수 있어 존재하는 것만 골라낸다.
metric_cols = [c for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
               if c in df_report.columns]
metric_means = df_report[metric_cols].mean(numeric_only=True)

# 텍스트 막대 그래프로 미리보기 — int(score*20) 으로 0~20 사이 막대 길이를 만든다.
print(f"\n{'='*52}")
print(f"Ragas 평가 결과 (메트릭별 평균)")
print(f"{'='*52}")
for metric, score in metric_means.items():
    bar = "#" * int(score * 20) + "." * (20 - int(score * 20))
    print(f"  {metric:<22} [{bar}] {score:.4f}")

## 5. 질문별 상세 점수

In [ ]:
df_eval = report.to_pandas()

display_cols = ["question", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]
available = [c for c in display_cols if c in df_eval.columns]
if available:
    print(df_eval[available].to_string(index=False))
else:
    print(df_eval.head(10).to_string())
df_eval

## 6. 시각화 — 4-패널 막대 차트

빨간 점선(0.7) 아래의 질문이 "개선이 필요한" 대상입니다.

In [ ]:
# 4 패널(2×2) 막대 차트로 메트릭별 점수를 한 화면에 그린다.
# 같은 데이터를 표 형태로도 봤지만, 시각화는 발표 슬라이드 캡쳐에 바로 쓸 수 있어 따로 그립니다.
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))   # 2행 × 2열 = 4개 서브플롯

metrics_list = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]   # 메트릭별 고유 색
titles = ["Faithfulness", "Answer Relevancy", "Context Precision", "Context Recall"]

# zip 으로 4개 리스트를 동시에 순회 — Pythonic 한 다중 반복 패턴.
# axes.flat 은 2D 배열을 1D 처럼 평탄하게 순회시킨다.
for ax, metric, color, title in zip(axes.flat, metrics_list, colors, titles):
    if metric not in df_eval.columns:
        ax.set_title(f"{title} (not available)")
        continue
    values = df_eval[metric].fillna(0)              # NaN 은 0으로 대체 → 차트가 깨지지 않게
    short_labels = [q[:18] + ("..." if len(q) > 18 else "") for q in df_eval["question"]]
    bars = ax.barh(range(len(values)), values, color=color, alpha=0.85)
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(short_labels, fontsize=8)
    ax.set_xlim(0, 1)                               # 모든 메트릭이 0~1 범위라 축 통일
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.axvline(x=0.7, color="red", linestyle="--", alpha=0.5)  # 빨간 점선 = 합격선 0.7
    ax.invert_yaxis()                               # 표와 같이 위에서 아래로 읽히도록 뒤집기
    # 막대 끝에 점수 라벨을 직접 그려서 색만 보고 안 보일 때도 값 확인 가능.
    for bar, val in zip(bars, values):
        ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2, f"{val:.2f}",
                va="center", fontsize=8)

plt.suptitle("Ragas Evaluation Report (v1)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("ragas_report.png", dpi=150, bbox_inches="tight")  # 발표용 PNG 로 저장
plt.show()
print("저장: ragas_report.png")

### 레이더 차트 — 전체 요약

4개 축이 균형 있게 바깥쪽으로 뻗어 있으면 좋은 에이전트. 한 축이 안쪽으로 들어가 있다면 그 메트릭이 약점입니다.

In [ ]:
# 레이더 차트 — 4 메트릭을 4 축으로 두고 한 다각형으로 그려 "전반적 균형"을 본다.
# 각 메트릭 평균값이 모두 바깥으로 멀리 뻗어 있으면 좋은 에이전트.
# `polar=True` 로 극좌표계를 사용합니다.
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# 평균 점수 4개를 추출. metric_means 에 키가 없으면 0 으로 폴백.
metric_values = [float(metric_means.get(m, 0)) for m in metrics_list]
# 레이더 차트는 마지막 점이 첫 점과 이어져야 해서, 시작값을 한 번 더 끝에 붙여 닫는다.
metric_values_closed = metric_values + [metric_values[0]]

# 0 ~ 2π 사이를 4 등분. endpoint=False → 한 바퀴를 다시 돌지 않게.
angles = np.linspace(0, 2 * np.pi, len(titles), endpoint=False).tolist()
angles_closed = angles + [angles[0]]   # 다각형이 닫히도록 마지막에 첫 각도 한 번 더

ax.plot(angles_closed, metric_values_closed, "o-", linewidth=2, color="#2196F3")
ax.fill(angles_closed, metric_values_closed, alpha=0.25, color="#2196F3")  # 안쪽 면적을 살짝 채움
ax.set_thetagrids(np.degrees(angles), titles)   # 라디안 → 도(degree) 변환해 라벨 표시
ax.set_ylim(0, 1)
ax.set_title("Agent Performance Radar (v1)", fontsize=12, fontweight="bold", pad=20)

# 각 점 옆에 숫자 라벨도 함께 표시
for angle, value in zip(angles, metric_values):
    ax.annotate(f"{value:.2f}", xy=(angle, value), fontsize=10, ha="center")

plt.tight_layout()
plt.savefig("ragas_radar.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. 낮은 점수 원인 진단

각 질문·메트릭을 스캔해 0.7 미만인 항목에 "의심 원인 + 처방" 을 자동 주석으로 답니다. 이 진단이 곧 튜닝 방향의 출발점입니다.

In [ ]:
DIAGNOSIS = {
    "faithfulness":       ("답변에 컨텍스트에 없는 정보가 포함됨 (할루시네이션)",
                           "프롬프트에 '주어진 결과만 사용' 명시, temperature=0 고정"),
    "answer_relevancy":   ("답변이 질문의 핵심에 직접 답하지 않음",
                           "프롬프트에 '질문에 직접 답하고 관련 없는 배경 생략' 지시"),
    "context_precision":  ("SQL 결과가 질문과 느슨하게 연결됨",
                           "SQL 생성 프롬프트에 Few-shot 추가, 테이블 선택 규칙 강화"),
    "context_recall":     ("정답에 필요한 사실이 SQL 결과에서 누락",
                           "스키마 COMMENT 보강, 쿼리 범위 확대(JOIN 추가), TopK↑"),
}

print("낮은 점수(<0.7) 케이스 진단\n")
flagged = 0
for _, row in df_eval.iterrows():
    low = [(m, row[m]) for m in metrics_list if m in row and pd.notna(row[m]) and row[m] < 0.7]
    if not low:
        continue
    flagged += 1
    print(f"Q: {row['question'][:50]}")
    for m, v in low:
        cause, rx = DIAGNOSIS[m]
        print(f"   - {m}={v:.2f}")
        print(f"       진단: {cause}")
        print(f"       처방: {rx}")
    print()

if flagged == 0:
    print("모든 메트릭이 0.7 이상 — 훌륭합니다. v2 튜닝은 생략 가능하지만, 교육 목적상 아래 셀을 그대로 진행하세요.")
else:
    print(f"총 {flagged}건 진단 완료 → 아래 v2 튜닝에서 Faithfulness 강화를 시도합니다.")

## 8. v2 튜닝 — 답변 프롬프트 개선

대표적인 개선 중 하나로 `generate_answer` 노드의 프롬프트를 **더 엄격하게** 바꿔 Faithfulness / Answer Relevancy 를 함께 노려봅니다. 아래 v2 프롬프트는:

- "**결과에 있는 정보만 사용하라**" 명시 (할루시네이션 억제 → Faithfulness↑)
- "**질문에 직접 답하라**" 지시 (Answer Relevancy↑)
- 숫자 천단위 구분 / 추측 금지 규칙 유지

> 튜닝은 **한 번에 한 변수만** 바꾸는 것이 원칙. 동시에 여러 개를 바꾸면 어떤 변경이 어떤 메트릭을 움직였는지 추적 불가능.

In [ ]:
ANSWER_TEMPLATE_V2 = ChatPromptTemplate.from_template(
    """아래 SQL 쿼리 결과를 바탕으로 질문에 답변하세요.

## 중요 규칙 (v2)
- 반드시 아래 결과에 있는 정보만 사용하세요.
- 결과에 없는 정보를 추가하거나 추측하지 마세요.
- 숫자는 천 단위 구분자를 사용하세요.
- 질문의 핵심에 직접 답변하세요. 관련 없는 배경 설명은 생략.

## 질문
{question}

## 실행된 SQL
{sql}

## 쿼리 결과
{result_md}

## 답변"""
)

answer_chain_v2 = ANSWER_TEMPLATE_V2 | answer_llm | StrOutputParser()


def generate_answer_v2(state: AgentState) -> dict:
    if state.get("error") and state.get("retry_count", 0) >= 3:
        return {
            "answer": (
                "죄송합니다. 질문에 답변하지 못했습니다.\n"
                f"- 마지막 에러: {state['error']}\n"
                "- 질문을 더 구체적으로 다시 물어봐 주세요."
            )
        }
    ans = answer_chain_v2.invoke({
        "question":  state.get("question", ""),
        "sql":       state.get("sql", ""),
        "result_md": (state.get("result_md") or "(결과 없음)")[:1500],
    })
    return {"answer": ans.strip()}


# v2 그래프 — answer 노드만 교체
graph_v2 = StateGraph(AgentState)
graph_v2.add_node("generate_sql", generate_sql)
graph_v2.add_node("execute_sql", execute_sql)
graph_v2.add_node("validate_sql", validate_sql)
graph_v2.add_node("generate_answer", generate_answer_v2)

graph_v2.set_entry_point("generate_sql")
graph_v2.add_edge("generate_sql", "execute_sql")
graph_v2.add_edge("execute_sql", "validate_sql")
graph_v2.add_conditional_edges(
    "validate_sql",
    should_retry,
    {
        "answer":  "generate_answer",
        "giveup":  "generate_answer",
        "retry":   "generate_sql",
    },
)
graph_v2.add_edge("generate_answer", END)

agent_v2 = graph_v2.compile()
print("v2 agent compiled (answer 노드만 교체).")

In [ ]:
# v2 실행 + eval_data 재구성
results_v2 = []
for i, q in enumerate(questions):
    state = agent_v2.invoke(_initial_state(q))
    results_v2.append({
        "question":  q,
        "sql":       state.get("sql", ""),
        "result_md": state.get("result_md", ""),
        "answer":    state.get("answer", ""),
        "error":     state.get("error", ""),
        "retries":   state.get("retry_count", 0),
    })
    print(f"[v2 {i+1:2d}/{len(questions)}] retries={state.get('retry_count',0)} — {q}")

eval_data_v2 = _build_eval_data(results_v2)
print(f"\nv2 결과 {len(results_v2)}건 수집 완료.")

In [ ]:
print("v2 Ragas 평가 실행 중... (약 2~5분)")
report_v2 = evaluate(
    Dataset.from_dict(eval_data_v2),
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm,
    embeddings=judge_emb,
)

df_report_v2 = report_v2.to_pandas()
metric_means_v2 = df_report_v2[[c for c in metric_cols if c in df_report_v2.columns]].mean(numeric_only=True)

print(f"\n{'='*52}")
print(f"v2 Ragas 평가 결과 (메트릭별 평균)")
print(f"{'='*52}")
for metric, score in metric_means_v2.items():
    bar = "#" * int(score * 20) + "." * (20 - int(score * 20))
    print(f"  {metric:<22} [{bar}] {score:.4f}")

## 9. Before / After 비교

숫자 표와 나란히 놓인 막대 차트 두 가지 포맷으로 뽑습니다. **막대 차트는 발표 슬라이드 3번째 장의 핵심 시각 자료**로 사용하세요.

In [ ]:
print(f"{'='*58}")
print(f"Before / After 비교 (v1 → v2)")
print(f"{'='*58}")
print(f"{'메트릭':<22} {'v1':>10} {'v2':>10} {'diff':>10}")
print("-" * 58)

compare_rows = []
for m in metrics_list:
    v1 = float(metric_means.get(m, 0))
    v2 = float(metric_means_v2.get(m, 0))
    diff = v2 - v1
    arrow = "UP  " if diff > 0.005 else ("DOWN" if diff < -0.005 else "SAME")
    print(f"  {m:<20} {v1:>10.4f} {v2:>10.4f}  {arrow} {diff:>+.4f}")
    compare_rows.append({"metric": m, "v1": v1, "v2": v2, "diff": diff})

pd.DataFrame(compare_rows)

In [ ]:
# Before/After 비교 차트 — 발표 슬라이드 3장째 핵심 자료입니다.
# 같은 4 메트릭을 v1(연한 파랑) / v2(진한 파랑) 두 막대로 나란히 그려 변화를 한눈에 보여 줍니다.
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_list))   # 0,1,2,3 — 메트릭 인덱스
width = 0.35                        # 막대 폭. 0.5 보다 작아야 두 막대가 옆으로 들어감

# 평균 점수만 비교 — 질문별 막대를 다 그리면 너무 복잡해서 평균 4개에 집중.
v1_scores = [float(metric_means.get(m, 0)) for m in metrics_list]
v2_scores = [float(metric_means_v2.get(m, 0)) for m in metrics_list]

# x - width/2 / x + width/2 → 같은 인덱스 자리에 막대 두 개를 좌우로 배치.
bars1 = ax.bar(x - width / 2, v1_scores, width, label="v1 (Before)", color="#90CAF9", edgecolor="white")
bars2 = ax.bar(x + width / 2, v2_scores, width, label="v2 (After)",  color="#2196F3", edgecolor="white")

ax.set_ylabel("Score")
ax.set_title("Ragas Evaluation — Before vs After Tuning")
ax.set_xticks(x)
ax.set_xticklabels(titles)
ax.set_ylim(0, 1)
ax.axhline(y=0.7, color="red", linestyle="--", alpha=0.5, label="Target 0.7")  # 합격선
ax.legend()

# 각 막대 위에 숫자 라벨을 직접 그려서 발표 시 굳이 표를 띄우지 않아도 점수가 보이게.
for bars in (bars1, bars2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("ragas_before_after.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장: ragas_before_after.png  (발표 슬라이드 3장째 핵심 차트)")

## 실습 과제

1. **본인 에이전트 Ragas 평가**: 위 코드의 `questions` / `ground_truths_dict` 를 본인 도메인 질문·정답으로 교체 후 평가를 실행하세요. 처음에는 3~5개로 축소해 파이프라인을 확인하고, 정상이면 10개로 확장합니다.
2. **최저 점수 진단**: `df_eval` 에서 가장 낮은 점수 질문 1건을 골라 SQL·답변·ground_truth 를 나란히 놓고 원인을 한 문단으로 정리하세요 (할루시네이션? SQL 오류? 프롬프트 모호?).
3. **v2 튜닝 선택**: 위 예시는 `answer` 프롬프트를 바꿨습니다. 본인 분석에 맞게 **한 가지만** 바꾸세요: 예를 들어 Context Recall 이 낮다면 `generate_sql` 프롬프트에 도메인 규칙·Few-shot 을 추가.
4. **Before/After 차트 캡처**: `ragas_before_after.png` 를 발표 슬라이드 3장째에 삽입하고, 캡션에 "어떤 변경이 어떤 메트릭을 개선/악화 시켰는가" 를 1~2줄로 씁니다.

In [ ]:
# ============================================================
# TODO — 본인 프로젝트에 Ragas 평가 적용
# ============================================================
# 1) questions / ground_truths_dict 교체
# my_questions = [
#     "...",
# ]
# my_ground_truth = {
#     "...": "...",
# }
#
# 2) 본인 에이전트 실행 → results 수집
# my_results = []
# for q in my_questions:
#     state = agent.invoke(_initial_state(q))
#     my_results.append({
#         "question":  q,
#         "sql":       state.get("sql", ""),
#         "result_md": state.get("result_md", ""),
#         "answer":    state.get("answer", ""),
#     })
#
# 3) 평가
# my_eval = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
# for r in my_results:
#     my_eval["question"].append(r["question"])
#     my_eval["answer"].append(r["answer"])
#     my_eval["contexts"].append([f"SQL: {r['sql']}\n결과:\n{r['result_md']}"])
#     my_eval["ground_truth"].append(my_ground_truth.get(r["question"], ""))
# my_report = evaluate(Dataset.from_dict(my_eval),
#                      metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
#                      llm=judge_llm, embeddings=judge_emb)
# print(my_report.to_pandas())
pass

## 다음 노트북에서는...

**더 이상 노트북은 없습니다.** 23H는 최종 튜닝 & 리허설 시간(개인 작업 + 강사 1:1 피드백), 24H는 최종 발표 & 수료식입니다. 준비물은:

- **에이전트 v1** — 17H·18H·19H 에서 다듬은 최종 버전.
- **LangSmith Trace URL** — 18H 에서 기록된 본인 프로젝트 링크.
- **Ragas 리포트** — `ragas_report.png` + `ragas_before_after.png` + 숫자 표.
- **발표 슬라이드 3장** — 문제 정의 / 아키텍처 / 결과 & 회고.

최종 발표 5~7분. 라이브 데모 2~4개 질문 + Ragas Before/After 차트 공유가 핵심입니다. 실패 사례와 디버깅 과정을 당당히 설명하세요 — 그것이 점수의 절반입니다.